# Survey Logic PoC - event-injected perimeter survey

This notebook models the survey as a deterministic state machine driven by semantic sensor events. The goal is not to make the mock court lucky; the goal is to define the same event contract that can later be produced by Gazebo sensors or real LiDAR/OAK-D/odom adapters.


## Core idea

The PoC has four layers:

1. **Event injection / sensor adapter contract** - raw virtual or real sensors become semantic events such as `near_net`, `near_fence`, `corner_detected`, and `turn_complete`.
2. **Survey sections** - each section says exactly what it is doing, which event it waits for, and what route point it records.
3. **Observed route** - the survey records points in traversal order, without yet claiming they are a valid court model.
4. **Canonical conversion** - observed fence/corner points are identified as named fences and named corners, then validated.

Important correction: after reaching the near baseline fence, the next drive section must traverse the whole baseline-side fence to the upper-left / near-left corner. It must not stop early just because a short-side threshold fired.


In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from enum import Enum
from typing import Iterable


## Normalized contracts

The state machine consumes `SensorEvent` objects. A real implementation would have separate adapters for Gazebo and hardware, but both adapters must emit the same event vocabulary.


In [ ]:
@dataclass(frozen=True)
class Point:
    x_m: float
    y_m: float

    def distance_to(self, other: "Point") -> float:
        return math.hypot(self.x_m - other.x_m, self.y_m - other.y_m)


@dataclass(frozen=True)
class Pose:
    x_m: float
    y_m: float
    yaw_rad: float


@dataclass(frozen=True)
class BaseCommand:
    linear_m_s: float = 0.0
    angular_rad_s: float = 0.0


class EventType(str, Enum):
    NEAR_NET = "near_net"
    NEAR_FENCE = "near_fence"
    CORNER_DETECTED = "corner_detected"
    TURN_COMPLETE = "turn_complete"
    LOOP_CLOSED = "loop_closed"
    COLLISION_RISK = "collision_risk"
    TIMEOUT = "timeout"


@dataclass(frozen=True)
class SensorEvent:
    kind: EventType
    pose: Pose
    confidence: float = 1.0
    source: str = "injected"
    label_hint: str | None = None
    details: dict[str, object] = field(default_factory=dict)


@dataclass(frozen=True)
class RoutePoint:
    label: str
    point: Point
    source_event: EventType
    confidence: float
    note: str = ""


## Section explanations

Each section has a plain-language explanation. These descriptions are deliberately part of the PoC because the route is easy to misunderstand: some sections drive toward a standoff point, while others follow an entire fence segment until a corner event arrives.


In [ ]:
class SurveyState(str, Enum):
    NET_STANDOFF = "net_standoff"
    TURN_TO_NEAR_BASELINE = "turn_to_near_baseline"
    DRIVE_TO_NEAR_BASELINE = "drive_to_near_baseline"
    TURN_TO_NEAR_BASELINE_FENCE = "turn_to_near_baseline_fence"
    FOLLOW_NEAR_BASELINE_TO_LEFT_CORNER = "follow_near_baseline_to_left_corner"
    TURN_TO_LEFT_SIDE_FENCE = "turn_to_left_side_fence"
    FOLLOW_LEFT_SIDE_TO_FAR_CORNER = "follow_left_side_to_far_corner"
    TURN_TO_FAR_BASELINE_FENCE = "turn_to_far_baseline_fence"
    FOLLOW_FAR_BASELINE_TO_RIGHT_CORNER = "follow_far_baseline_to_right_corner"
    TURN_TO_RIGHT_SIDE_FENCE = "turn_to_right_side_fence"
    FOLLOW_RIGHT_SIDE_TO_NEAR_RIGHT_CORNER = "follow_right_side_to_near_right_corner"
    FOLLOW_NEAR_BASELINE_TO_LOOP_CLOSE = "follow_near_baseline_to_loop_close"
    ROUTE_COMPLETED = "route_completed"
    FAILED = "failed"


SECTION_EXPLANATIONS: dict[SurveyState, str] = {
    SurveyState.NET_STANDOFF:
        "Drive toward the net until the sensor adapter emits near_net. Record only the net reference/standoff point.",
    SurveyState.TURN_TO_NEAR_BASELINE:
        "Turn 180 degrees away from the net. Wait for turn_complete before driving.",
    SurveyState.DRIVE_TO_NEAR_BASELINE:
        "Drive away from the net until near_fence identifies the near baseline fence standoff.",
    SurveyState.TURN_TO_NEAR_BASELINE_FENCE:
        "Turn left to align with the near baseline fence corridor.",
    SurveyState.FOLLOW_NEAR_BASELINE_TO_LEFT_CORNER:
        "Follow the whole near-baseline fence side until the upper-left / near-left corner is detected. This is the first corrected failure point.",
    SurveyState.TURN_TO_LEFT_SIDE_FENCE:
        "Turn left to align with the left side fence.",
    SurveyState.FOLLOW_LEFT_SIDE_TO_FAR_CORNER:
        "Follow the left side fence all the way to the far-left corner.",
    SurveyState.TURN_TO_FAR_BASELINE_FENCE:
        "Turn left to align with the far baseline fence.",
    SurveyState.FOLLOW_FAR_BASELINE_TO_RIGHT_CORNER:
        "Follow the far baseline fence to the far-right corner.",
    SurveyState.TURN_TO_RIGHT_SIDE_FENCE:
        "Turn left to align with the right side fence.",
    SurveyState.FOLLOW_RIGHT_SIDE_TO_NEAR_RIGHT_CORNER:
        "Follow the right side fence down to the near-right corner.",
    SurveyState.FOLLOW_NEAR_BASELINE_TO_LOOP_CLOSE:
        "Follow the near baseline fence back toward the first near-left corner and require loop closure.",
}

for state, explanation in SECTION_EXPLANATIONS.items():
    print(f"{state.value}: {explanation}")


## Section implementations

The sections do not inspect raw LiDAR rays or camera frames. They only consume events. That keeps this PoC honest: later, the hard part is making the adapters emit reliable events from real or virtual sensors.


In [ ]:
@dataclass(frozen=True)
class SectionResult:
    command: BaseCommand
    done: bool = False
    recorded_point: RoutePoint | None = None
    failure_reason: str | None = None
    event: str = ""


class SurveySection:
    def __init__(
        self,
        state: SurveyState,
        next_state: SurveyState,
        expected_event: EventType,
        record_label: str | None = None,
        min_confidence: float = 0.75,
    ) -> None:
        self.state = state
        self.next_state = next_state
        self.expected_event = expected_event
        self.record_label = record_label
        self.min_confidence = min_confidence

    @property
    def explanation(self) -> str:
        return SECTION_EXPLANATIONS.get(self.state, "")

    def process(self, event: SensorEvent | None) -> SectionResult:
        if event is None:
            return SectionResult(BaseCommand(0.25, 0.0), event="waiting_for_event")
        if event.kind == EventType.COLLISION_RISK:
            return SectionResult(BaseCommand(), failure_reason="collision_risk_before_avoidance_layer")
        if event.kind == EventType.TIMEOUT:
            return SectionResult(BaseCommand(), failure_reason=f"timeout_in_{self.state.value}")
        if event.kind != self.expected_event:
            return SectionResult(BaseCommand(0.15, 0.0), event=f"ignored_{event.kind.value}")
        if event.confidence < self.min_confidence:
            return SectionResult(BaseCommand(0.0, 0.15), event="event_confidence_too_low")

        point = None
        if self.record_label is not None:
            point = RoutePoint(
                label=self.record_label,
                point=Point(event.pose.x_m, event.pose.y_m),
                source_event=event.kind,
                confidence=event.confidence,
                note=event.label_hint or "",
            )
        return SectionResult(BaseCommand(), done=True, recorded_point=point, event=f"accepted_{event.kind.value}")


class FollowFenceToCornerSection(SurveySection):
    def __init__(self, state: SurveyState, next_state: SurveyState, corner_label: str, expected_hint: str) -> None:
        super().__init__(state, next_state, EventType.CORNER_DETECTED, record_label=corner_label)
        self.expected_hint = expected_hint

    def process(self, event: SensorEvent | None) -> SectionResult:
        result = super().process(event)
        if result.done and event is not None and event.label_hint != self.expected_hint:
            return SectionResult(
                BaseCommand(),
                failure_reason=f"expected_{self.expected_hint}_but_got_{event.label_hint}",
            )
        return result


## Survey orchestrator

The orchestrator advances only when a section accepts its expected event. It records observed route points but does not claim the court is valid; that belongs to canonical conversion.


In [ ]:
class EventInjectedSurvey:
    def __init__(self) -> None:
        self.sections = self._build_sections()
        self.index = 0
        self.route: list[RoutePoint] = []
        self.failed_reason: str | None = None

    def _build_sections(self) -> list[SurveySection]:
        return [
            SurveySection(SurveyState.NET_STANDOFF, SurveyState.TURN_TO_NEAR_BASELINE, EventType.NEAR_NET, "near_net_standoff"),
            SurveySection(SurveyState.TURN_TO_NEAR_BASELINE, SurveyState.DRIVE_TO_NEAR_BASELINE, EventType.TURN_COMPLETE),
            SurveySection(SurveyState.DRIVE_TO_NEAR_BASELINE, SurveyState.TURN_TO_NEAR_BASELINE_FENCE, EventType.NEAR_FENCE, "near_baseline_fence_standoff"),
            SurveySection(SurveyState.TURN_TO_NEAR_BASELINE_FENCE, SurveyState.FOLLOW_NEAR_BASELINE_TO_LEFT_CORNER, EventType.TURN_COMPLETE),
            FollowFenceToCornerSection(SurveyState.FOLLOW_NEAR_BASELINE_TO_LEFT_CORNER, SurveyState.TURN_TO_LEFT_SIDE_FENCE, "near_left_fence_corner", "near_left_corner"),
            SurveySection(SurveyState.TURN_TO_LEFT_SIDE_FENCE, SurveyState.FOLLOW_LEFT_SIDE_TO_FAR_CORNER, EventType.TURN_COMPLETE),
            FollowFenceToCornerSection(SurveyState.FOLLOW_LEFT_SIDE_TO_FAR_CORNER, SurveyState.TURN_TO_FAR_BASELINE_FENCE, "far_left_fence_corner", "far_left_corner"),
            SurveySection(SurveyState.TURN_TO_FAR_BASELINE_FENCE, SurveyState.FOLLOW_FAR_BASELINE_TO_RIGHT_CORNER, EventType.TURN_COMPLETE),
            FollowFenceToCornerSection(SurveyState.FOLLOW_FAR_BASELINE_TO_RIGHT_CORNER, SurveyState.TURN_TO_RIGHT_SIDE_FENCE, "far_right_fence_corner", "far_right_corner"),
            SurveySection(SurveyState.TURN_TO_RIGHT_SIDE_FENCE, SurveyState.FOLLOW_RIGHT_SIDE_TO_NEAR_RIGHT_CORNER, EventType.TURN_COMPLETE),
            FollowFenceToCornerSection(SurveyState.FOLLOW_RIGHT_SIDE_TO_NEAR_RIGHT_CORNER, SurveyState.FOLLOW_NEAR_BASELINE_TO_LOOP_CLOSE, "near_right_fence_corner", "near_right_corner"),
            SurveySection(SurveyState.FOLLOW_NEAR_BASELINE_TO_LOOP_CLOSE, SurveyState.ROUTE_COMPLETED, EventType.LOOP_CLOSED, "near_left_loop_closure"),
        ]

    @property
    def state(self) -> SurveyState:
        if self.failed_reason:
            return SurveyState.FAILED
        if self.index >= len(self.sections):
            return SurveyState.ROUTE_COMPLETED
        return self.sections[self.index].state

    @property
    def current_explanation(self) -> str:
        if self.index >= len(self.sections):
            return "Route event sequence completed; canonical validation still required."
        return self.sections[self.index].explanation

    def update(self, event: SensorEvent | None) -> SectionResult:
        if self.state in {SurveyState.ROUTE_COMPLETED, SurveyState.FAILED}:
            return SectionResult(BaseCommand())
        section = self.sections[self.index]
        result = section.process(event)
        if result.failure_reason:
            self.failed_reason = result.failure_reason
            return result
        if result.recorded_point is not None:
            self.route.append(result.recorded_point)
        if result.done:
            self.index += 1
        return result


## Realistic event injection scenario

This scenario represents the corrected intended route. Notice that the first corner after the near baseline standoff is `near_left_corner`, reached by traversing the whole near-baseline fence side.


In [ ]:
def corrected_perimeter_events() -> list[SensorEvent]:
    return [
        SensorEvent(EventType.NEAR_NET, Pose(0.0, -1.6, math.pi / 2), label_hint="net_standoff"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(0.0, -1.6, -math.pi / 2)),
        SensorEvent(EventType.NEAR_FENCE, Pose(0.0, -13.8, -math.pi / 2), label_hint="near_baseline_fence"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(0.0, -13.8, math.pi)),
        SensorEvent(EventType.CORNER_DETECTED, Pose(-7.4, -13.8, math.pi), label_hint="near_left_corner"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(-7.4, -13.8, math.pi / 2)),
        SensorEvent(EventType.CORNER_DETECTED, Pose(-7.4, 13.8, math.pi / 2), label_hint="far_left_corner"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(-7.4, 13.8, 0.0)),
        SensorEvent(EventType.CORNER_DETECTED, Pose(7.4, 13.8, 0.0), label_hint="far_right_corner"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(7.4, 13.8, -math.pi / 2)),
        SensorEvent(EventType.CORNER_DETECTED, Pose(7.4, -13.8, -math.pi / 2), label_hint="near_right_corner"),
        SensorEvent(EventType.LOOP_CLOSED, Pose(-7.4, -13.8, math.pi), label_hint="near_left_corner"),
    ]


def early_corner_failure_events() -> list[SensorEvent]:
    events = corrected_perimeter_events()
    events[4] = SensorEvent(EventType.CORNER_DETECTED, Pose(1.2, -13.8, math.pi), label_hint="near_left_corner")
    return events


def run_survey(events: Iterable[SensorEvent]) -> EventInjectedSurvey:
    survey = EventInjectedSurvey()
    for event in events:
        before = survey.state
        result = survey.update(event)
        print(f"{before.value:42s} + {event.kind.value:16s} -> {survey.state.value:42s} {result.event or result.failure_reason or ''}")
        if survey.failed_reason:
            break
    return survey


survey = run_survey(corrected_perimeter_events())
print("\nObserved route:")
for point in survey.route:
    print(f"{point.label:32s} ({point.point.x_m:6.2f}, {point.point.y_m:6.2f})  {point.note}")


## Canonical conversion

The canonical converter turns observed route points into identified fence/corner entities. This is where raw points become semantic knowledge: near baseline fence, left side fence, far baseline fence, right side fence, and the named corners connecting them.


In [ ]:
@dataclass(frozen=True)
class IdentifiedFence:
    label: str
    start_corner: str
    end_corner: str
    start: Point
    end: Point

    @property
    def length_m(self) -> float:
        return self.start.distance_to(self.end)


@dataclass(frozen=True)
class CanonicalFenceModel:
    status: str
    corners: dict[str, Point]
    fences: dict[str, IdentifiedFence]
    reference_points: dict[str, Point]
    validation_errors: list[str]


class CanonicalConverter:
    REQUIRED = {
        "near_net_standoff",
        "near_baseline_fence_standoff",
        "near_left_fence_corner",
        "far_left_fence_corner",
        "far_right_fence_corner",
        "near_right_fence_corner",
        "near_left_loop_closure",
    }

    def __init__(self, min_baseline_half_traverse_m: float = 4.0, loop_tolerance_m: float = 0.75) -> None:
        self.min_baseline_half_traverse_m = min_baseline_half_traverse_m
        self.loop_tolerance_m = loop_tolerance_m

    def convert(self, route: list[RoutePoint]) -> CanonicalFenceModel:
        by_label = {p.label: p.point for p in route}
        errors = []
        missing = sorted(self.REQUIRED - set(by_label))
        if missing:
            errors.append(f"missing_route_points:{','.join(missing)}")

        self._validate_near_left_prefix(by_label, errors)

        corners = {}
        if not missing:
            corners = {
                "near_left": by_label["near_left_fence_corner"],
                "far_left": by_label["far_left_fence_corner"],
                "far_right": by_label["far_right_fence_corner"],
                "near_right": by_label["near_right_fence_corner"],
            }
            if by_label["near_left_loop_closure"].distance_to(corners["near_left"]) > self.loop_tolerance_m:
                errors.append("loop_closure_miss")
            errors.extend(self._validate_rectangle(corners))

        fences = {}
        if corners:
            fences = {
                "near_baseline_fence": IdentifiedFence("near_baseline_fence", "near_left", "near_right", corners["near_left"], corners["near_right"]),
                "left_side_fence": IdentifiedFence("left_side_fence", "near_left", "far_left", corners["near_left"], corners["far_left"]),
                "far_baseline_fence": IdentifiedFence("far_baseline_fence", "far_left", "far_right", corners["far_left"], corners["far_right"]),
                "right_side_fence": IdentifiedFence("right_side_fence", "far_right", "near_right", corners["far_right"], corners["near_right"]),
            }

        references = {k: v for k, v in by_label.items() if k in {"near_net_standoff", "near_baseline_fence_standoff"}}
        return CanonicalFenceModel(
            status="VALID" if not errors else "PARTIAL",
            corners=corners,
            fences=fences,
            reference_points=references,
            validation_errors=errors,
        )

    def _validate_near_left_prefix(self, by_label: dict[str, Point], errors: list[str]) -> None:
        baseline = by_label.get("near_baseline_fence_standoff")
        near_left = by_label.get("near_left_fence_corner")
        if baseline is None or near_left is None:
            return
        dx = abs(near_left.x_m - baseline.x_m)
        dy = abs(near_left.y_m - baseline.y_m)
        if baseline.distance_to(near_left) < self.min_baseline_half_traverse_m:
            errors.append("near_baseline_to_left_corner_traverse_too_short")
        if dy > dx:
            errors.append("near_baseline_to_left_corner_axis_invalid")

    def _validate_rectangle(self, corners: dict[str, Point]) -> list[str]:
        errors = []
        ordered = [corners["near_left"], corners["far_left"], corners["far_right"], corners["near_right"]]
        lengths = [ordered[i].distance_to(ordered[(i + 1) % 4]) for i in range(4)]
        if min(lengths) < 3.0:
            errors.append("fence_segment_too_short")
        for i in range(4):
            a = ordered[i - 1]
            b = ordered[i]
            c = ordered[(i + 1) % 4]
            v1 = (a.x_m - b.x_m, a.y_m - b.y_m)
            v2 = (c.x_m - b.x_m, c.y_m - b.y_m)
            dot = v1[0] * v2[0] + v1[1] * v2[1]
            norm = math.hypot(*v1) * math.hypot(*v2)
            if norm and abs(dot / norm) > 0.20:
                errors.append("corner_not_approximately_orthogonal")
                break
        return errors


model = CanonicalConverter().convert(survey.route)
print("status:", model.status)
print("validation_errors:", model.validation_errors)
print("identified fences:")
for label, fence in model.fences.items():
    print(f"  {label:22s} {fence.start_corner:10s}->{fence.end_corner:10s} length={fence.length_m:5.2f}m")


## Waypoint-driven event interpretation

In the real survey, events do not simply mean "jump to the next state". Sensor events propose waypoints while the robot is travelling. The current section decides whether a detected waypoint belongs to the expected route. If accepted, it becomes the active waypoint; when the robot reaches it, the waypoint is committed as a route point and the section advances.

This matters for the current failure: the run proposes a `near_left_corner`, but the geometry says it is not actually the near-left fence corner. The robot then drives toward the next long-side target, but no valid `far_left_corner` waypoint is discovered.


In [ ]:
@dataclass(frozen=True)
class CandidateWaypoint:
    label: str
    point: Point
    source_event: SensorEvent
    section: SurveyState


class WaypointDrivenSurvey:
    def __init__(self) -> None:
        self.sections = EventInjectedSurvey()._build_sections()
        self.index = 0
        self.active_waypoint: CandidateWaypoint | None = None
        self.route: list[RoutePoint] = []
        self.log: list[str] = []
        self.failed_reason: str | None = None

    @property
    def state(self) -> SurveyState:
        if self.failed_reason:
            return SurveyState.FAILED
        if self.index >= len(self.sections):
            return SurveyState.ROUTE_COMPLETED
        return self.sections[self.index].state

    def observe(self, event: SensorEvent) -> None:
        if self.state in {SurveyState.ROUTE_COMPLETED, SurveyState.FAILED}:
            return
        section = self.sections[self.index]
        result = section.process(event)
        if result.failure_reason:
            self.failed_reason = result.failure_reason
            self.log.append(f"{section.state.value}: failed {result.failure_reason}")
            return
        if result.recorded_point is not None:
            self.active_waypoint = CandidateWaypoint(
                result.recorded_point.label,
                result.recorded_point.point,
                event,
                section.state,
            )
            self.log.append(
                f"{section.state.value}: event {event.kind.value}/{event.label_hint} added waypoint "
                f"{result.recorded_point.label} at ({event.pose.x_m:.2f}, {event.pose.y_m:.2f})"
            )
            return
        if result.done:
            self.index += 1
            self.log.append(f"{section.state.value}: event {event.kind.value} advanced section")
        else:
            self.log.append(f"{section.state.value}: event {event.kind.value}/{event.label_hint} ignored ({result.event})")

    def reach_active_waypoint(self) -> None:
        if self.active_waypoint is None:
            self.log.append(f"{self.state.value}: no active waypoint to reach")
            return
        waypoint = self.active_waypoint
        self.route.append(
            RoutePoint(
                label=waypoint.label,
                point=waypoint.point,
                source_event=waypoint.source_event.kind,
                confidence=waypoint.source_event.confidence,
                note=f"committed from {waypoint.source_event.source}",
            )
        )
        self.log.append(f"{waypoint.section.value}: reached active waypoint {waypoint.label}; section committed")
        self.active_waypoint = None
        self.index += 1


def waypoint_effect_demo_events() -> list[SensorEvent]:
    return [
        SensorEvent(EventType.NEAR_NET, Pose(0.0, 0.0, 0.0), source="real_run", label_hint="net_standoff"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(-6.21, 0.0, -math.pi / 2), source="real_run"),
        SensorEvent(EventType.NEAR_FENCE, Pose(-6.21, 0.0, -math.pi / 2), source="real_run", label_hint="near_baseline_fence"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(-6.34, -0.01, -math.pi), source="real_run"),
        SensorEvent(EventType.CORNER_DETECTED, Pose(-6.37, -4.46, -math.pi), source="real_run", label_hint="near_left_corner"),
        SensorEvent(EventType.TURN_COMPLETE, Pose(-6.37, -4.55, math.pi / 2), source="real_run"),
        # No far_left_corner event arrives after this point.
    ]


waypoint_survey = WaypointDrivenSurvey()
for event in waypoint_effect_demo_events():
    waypoint_survey.observe(event)
    if waypoint_survey.active_waypoint is not None:
        waypoint_survey.reach_active_waypoint()

waypoint_model = CanonicalConverter().convert(waypoint_survey.route)
print("Waypoint-driven log:")
for line in waypoint_survey.log:
    print("-", line)
print("\nCommitted route:", [p.label for p in waypoint_survey.route])
print("state waiting for:", waypoint_survey.state.value)
print("canonical_status:", waypoint_model.status)
print("validation_errors:", waypoint_model.validation_errors)
assert waypoint_survey.state == SurveyState.FOLLOW_LEFT_SIDE_TO_FAR_CORNER
assert "near_baseline_to_left_corner_axis_invalid" in waypoint_model.validation_errors
assert "missing_route_points" in waypoint_model.validation_errors[0]


## Visualization

The visualization draws the observed route points and the canonical identified fences. It is intentionally based on the canonical model, not on a hidden simulator, so it shows exactly what the converter believes the court/fence entities are.


In [ ]:
def _route_svg(route: list[RoutePoint], model: CanonicalFenceModel, title: str) -> str:
    import html

    points = [(p.point.x_m, p.point.y_m) for p in route]
    corners = [(p.x_m, p.y_m) for p in model.corners.values()]
    court_half_width_m = 10.97 / 2
    court_half_length_m = 23.77 / 2
    court_points = [
        (-court_half_width_m, -court_half_length_m),
        (court_half_width_m, -court_half_length_m),
        (court_half_width_m, court_half_length_m),
        (-court_half_width_m, court_half_length_m),
    ]
    all_points = points + corners + court_points
    min_x = min(x for x, _ in all_points) - 2.0
    max_x = max(x for x, _ in all_points) + 2.0
    min_y = min(y for _, y in all_points) - 2.0
    max_y = max(y for _, y in all_points) + 2.0

    width, height, margin = 920, 760, 72
    scale = min((width - 2 * margin) / max(1.0, max_x - min_x), (height - 2 * margin) / max(1.0, max_y - min_y))

    def sx(x: float) -> float:
        return margin + (x - min_x) * scale

    def sy(y: float) -> float:
        return height - margin - (y - min_y) * scale

    def polyline(coords: list[tuple[float, float]], close: bool = False) -> str:
        pts = " ".join(f"{sx(x):.1f},{sy(y):.1f}" for x, y in coords)
        if close and coords:
            pts += f" {sx(coords[0][0]):.1f},{sy(coords[0][1]):.1f}"
        return pts

    status_color = "#2e7d32" if model.status == "VALID" else "#c62828"
    court_pts = polyline(court_points, close=True)
    route_pts = polyline(points)
    errors = html.escape(", ".join(model.validation_errors) or "no validation errors")

    fence_lines = []
    for fence in model.fences.values():
        fence_lines.append(
            f'<line x1="{sx(fence.start.x_m):.1f}" y1="{sy(fence.start.y_m):.1f}" '
            f'x2="{sx(fence.end.x_m):.1f}" y2="{sy(fence.end.y_m):.1f}" '
            'stroke="#263238" stroke-width="4" stroke-linecap="round" />'
        )

    route_marks = []
    for idx, point in enumerate(route, start=1):
        x = sx(point.point.x_m)
        y = sy(point.point.y_m)
        route_marks.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="7" fill="#1976d2" stroke="#ffffff" stroke-width="2" />')
        route_marks.append(f'<text x="{x + 10:.1f}" y="{y - 10:.1f}" class="label">{idx}. {html.escape(point.label)}</text>')

    corner_marks = []
    for label, point in model.corners.items():
        x = sx(point.x_m)
        y = sy(point.y_m)
        corner_marks.append(f'<rect x="{x - 6:.1f}" y="{y - 6:.1f}" width="12" height="12" fill="{status_color}" stroke="#ffffff" stroke-width="2" />')
        corner_marks.append(f'<text x="{x + 10:.1f}" y="{y + 18:.1f}" class="corner">{html.escape(label)}</text>')

    reference_marks = []
    for label, point in model.reference_points.items():
        x = sx(point.x_m)
        y = sy(point.y_m)
        reference_marks.append(f'<polygon points="{x:.1f},{y-8:.1f} {x-8:.1f},{y+8:.1f} {x+8:.1f},{y+8:.1f}" fill="#f9a825" stroke="#ffffff" stroke-width="2" />')
        reference_marks.append(f'<text x="{x + 10:.1f}" y="{y + 18:.1f}" class="ref">{html.escape(label)}</text>')

    return f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
  <style>
    .title {{ font: 700 22px Arial, sans-serif; fill: #17202a; }}
    .subtitle {{ font: 14px Arial, sans-serif; fill: #4d5b65; }}
    .label {{ font: 12px Arial, sans-serif; fill: #0d47a1; }}
    .corner {{ font: 12px Arial, sans-serif; fill: {status_color}; font-weight: 700; }}
    .ref {{ font: 12px Arial, sans-serif; fill: #8d6e00; }}
    .legend {{ font: 13px Arial, sans-serif; fill: #263238; }}
  </style>
  <rect width="100%" height="100%" fill="#f8faf7" />
  <text x="36" y="38" class="title">{html.escape(title)}</text>
  <text x="36" y="61" class="subtitle">status={html.escape(model.status)} | {errors}</text>
  <polyline points="{court_pts}" fill="#e8f5e9" stroke="#8d6e63" stroke-width="2" stroke-dasharray="8 6" />
  <line x1="{sx(-court_half_width_m):.1f}" y1="{sy(0.0):.1f}" x2="{sx(court_half_width_m):.1f}" y2="{sy(0.0):.1f}" stroke="#111111" stroke-width="3" />
  <text x="{sx(court_half_width_m) + 10:.1f}" y="{sy(0.0) + 4:.1f}" class="legend">net</text>
  <text x="{sx(court_half_width_m) + 10:.1f}" y="{sy(court_half_length_m) + 4:.1f}" class="legend">playable outer lines</text>
  <polyline points="{route_pts}" fill="none" stroke="#1976d2" stroke-width="2.5" stroke-dasharray="6 5" />
  {''.join(fence_lines)}
  {''.join(route_marks)}
  {''.join(corner_marks)}
  {''.join(reference_marks)}
</svg>'''


def _display_svg(svg: str) -> object:
    try:
        from IPython.display import SVG, display
        obj = SVG(svg)
        display(obj)
        return obj
    except Exception:
        print(svg[:500] + "...")
        return svg


def visualize_canonical_model(
    route: list[RoutePoint],
    model: CanonicalFenceModel,
    title: str = "Survey route and canonical fence model",
) -> object | None:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        return _display_svg(_route_svg(route, model, title))

    fig, ax = plt.subplots(figsize=(9, 9))
    status_color = "#2e7d32" if model.status == "VALID" else "#c62828"
    court_half_width_m = 10.97 / 2
    court_half_length_m = 23.77 / 2

    court_x = [-court_half_width_m, court_half_width_m, court_half_width_m, -court_half_width_m, -court_half_width_m]
    court_y = [-court_half_length_m, -court_half_length_m, court_half_length_m, court_half_length_m, -court_half_length_m]
    ax.plot(court_x, court_y, color="#8d6e63", linewidth=1.5, linestyle="--", alpha=0.9, label="playable outer lines")
    ax.plot([-court_half_width_m, court_half_width_m], [0.0, 0.0], color="#111111", linewidth=2.5, label="net")
    ax.annotate("net", (0.0, 0.0), textcoords="offset points", xytext=(8, 8), fontsize=9, color="#111111")

    if model.fences:
        for fence in model.fences.values():
            ax.plot([fence.start.x_m, fence.end.x_m], [fence.start.y_m, fence.end.y_m], linewidth=3, color="#263238", alpha=0.85)
            mx = (fence.start.x_m + fence.end.x_m) / 2
            my = (fence.start.y_m + fence.end.y_m) / 2
            ax.annotate(fence.label.replace("_", "\n"), (mx, my), textcoords="offset points", xytext=(6, 6), fontsize=8, color="#263238")

    if route:
        xs = [p.point.x_m for p in route]
        ys = [p.point.y_m for p in route]
        ax.plot(xs, ys, "--", color="#1976d2", linewidth=1.5, alpha=0.7, label="observed route order")
        ax.scatter(xs, ys, s=42, color="#1976d2", zorder=3)
        for idx, point in enumerate(route, start=1):
            ax.annotate(f"{idx}. {point.label.replace('_', ' ')}", (point.point.x_m, point.point.y_m), textcoords="offset points", xytext=(8, -12), fontsize=7, color="#0d47a1")

    if model.corners:
        for label, point in model.corners.items():
            ax.scatter([point.x_m], [point.y_m], s=110, marker="s", color=status_color, zorder=4)
            ax.annotate(label, (point.x_m, point.y_m), textcoords="offset points", xytext=(8, 8), fontsize=9, weight="bold", color=status_color)

    if model.reference_points:
        for label, point in model.reference_points.items():
            ax.scatter([point.x_m], [point.y_m], s=80, marker="^", color="#f9a825", zorder=5)
            ax.annotate(label.replace("_", "\n"), (point.x_m, point.y_m), textcoords="offset points", xytext=(8, 8), fontsize=8, color="#8d6e00")

    subtitle = f"status={model.status}"
    if model.validation_errors:
        subtitle += " | " + ", ".join(model.validation_errors)
    ax.set_title(f"{title}\n{subtitle}", color=status_color)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.tight_layout()
    return fig


valid_fig = visualize_canonical_model(survey.route, model, "Corrected injected route")


## Replay events from a real Map Court run

This section uses the same injectable PoC interface, but the inputs come from a recorded Map Court JSONL run instead of the handcrafted happy path. The replay adapter preserves the runtime event names in `source/details`, maps them to PoC `SensorEvent` kinds, and lets the section FSM decide what is accepted or ignored.

Expected observation for the known failing fixture: the survey reaches the near-left corner, then stalls before far-left/far-right/near-right are identified. That should produce a `PARTIAL` canonical model and a partial visualization, not a successful rectangle.


In [ ]:
import json
from pathlib import Path


def _workspace_root() -> Path:
    here = Path.cwd()
    if (here / "fixtures").exists():
        return here
    if (here.parent / "fixtures").exists():
        return here.parent
    return here


RUNTIME_EVENT_TO_POC: dict[str, list[tuple[EventType, str | None]]] = {
    # Runtime transition events are often compound. Expand them into the atomic
    # injectable events that the PoC section FSM expects.
    "net_found": [(EventType.NEAR_NET, "net_standoff")],
    "net_standoff_reached": [(EventType.NEAR_NET, "net_standoff")],
    "baseline_found": [(EventType.TURN_COMPLETE, None), (EventType.NEAR_FENCE, "near_baseline_fence")],
    "baseline_fence_reached": [(EventType.TURN_COMPLETE, None), (EventType.NEAR_FENCE, "near_baseline_fence")],
    "turn_180_complete": [(EventType.TURN_COMPLETE, None)],
    "aligned_for_sideline_drive": [(EventType.TURN_COMPLETE, None)],
    "side_fence_reached": [(EventType.CORNER_DETECTED, "near_left_corner")],
    "aligned_for_long_side_drive": [(EventType.TURN_COMPLETE, None)],
    "far_baseline_fence_reached": [(EventType.CORNER_DETECTED, "far_left_corner")],
    "aligned_for_far_short_drive": [(EventType.TURN_COMPLETE, None)],
    "far_side_fence_reached": [(EventType.CORNER_DETECTED, "far_right_corner")],
    "aligned_for_return_drive": [(EventType.TURN_COMPLETE, None)],
    "return_baseline_fence_reached": [(EventType.CORNER_DETECTED, "near_right_corner")],
    "full_perimeter_survey_success": [(EventType.LOOP_CLOSED, "near_left_corner")],
}


def _map_runtime_event(raw_event: str) -> list[tuple[EventType, str | None]]:
    for prefix, mapped in RUNTIME_EVENT_TO_POC.items():
        if raw_event.startswith(prefix):
            return mapped
    return []


def replay_events_from_run(path: Path) -> list[SensorEvent]:
    events: list[SensorEvent] = []
    seen_raw_events: set[str] = set()
    for tick, line in enumerate(path.read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        row = json.loads(line)
        raw_event = row.get("survey_event") or (row.get("navigation") or {}).get("last_event")
        if not raw_event or raw_event in seen_raw_events:
            continue
        mapped_events = _map_runtime_event(raw_event)
        if not mapped_events:
            continue
        seen_raw_events.add(raw_event)
        for ordinal, (kind, label_hint) in enumerate(mapped_events):
            events.append(
                SensorEvent(
                    kind=kind,
                    pose=Pose(float(row.get("x_m") or 0.0), float(row.get("y_m") or 0.0), float(row.get("yaw_rad") or 0.0)),
                    confidence=1.0,
                    source="map_court_replay",
                    label_hint=label_hint,
                    details={"runtime_event": raw_event, "tick": tick, "expanded_index": ordinal},
                )
            )
    return events


replay_path = _workspace_root() / "fixtures" / "navigation" / "survey" / "long_side_inside_court_timeout_2026-06-05.jsonl"
real_run_events = replay_events_from_run(replay_path)
print("Injected atomic events expanded from run:")
for event in real_run_events:
    print(
        f"tick={event.details['tick']:4d}.{event.details['expanded_index']} {event.details['runtime_event']:40s} -> "
        f"{event.kind.value:16s} hint={event.label_hint} pose=({event.pose.x_m:.2f}, {event.pose.y_m:.2f})"
    )

real_run_survey = run_survey(real_run_events)
real_run_model = CanonicalConverter().convert(real_run_survey.route)
print("\nAccepted route points from run:")
for point in real_run_survey.route:
    print(f"{point.label:32s} ({point.point.x_m:6.2f}, {point.point.y_m:6.2f})")
print("\nfinal_poc_state:", real_run_survey.state.value)
print("canonical_status:", real_run_model.status)
print("validation_errors:", real_run_model.validation_errors)
assert real_run_model.status == "PARTIAL"
assert "missing_route_points" in real_run_model.validation_errors[0]
assert [p.label for p in real_run_survey.route] == [
    "near_net_standoff",
    "near_baseline_fence_standoff",
    "near_left_fence_corner",
]

real_run_fig = visualize_canonical_model(real_run_survey.route, real_run_model, "Real Map Court replay events")


## Runtime-compatible survey result

The runtime and dashboard already expect the completed survey to look like `runtime/court_boundary.json`.

Required consumers found in the codebase:

- `LidarSurveyBoundaryProvider` reads `survey_complete` and derives operational bounds from `canonical_fence_model.corners`.
- `TennisRobotDB.import_survey(...)` imports `canonical_fence_model`, `boundary_distances`, `is_doubles`, `court_geometry`, `survey_type`, and `geometry.net_world_pos`.

So the PoC result below converts the canonical model into the same shape instead of inventing a separate result contract.


In [ ]:
@dataclass(frozen=True)
class CourtLineModel:
    length_m: float = 23.77
    doubles_width_m: float = 10.97
    singles_width_m: float = 8.23
    width_tolerance_m: float = 0.35


def build_runtime_survey_result(
    model: CanonicalFenceModel,
    route: list[RoutePoint],
    court_lines: CourtLineModel = CourtLineModel(),
    surveyed_at: float = 1780450000.0,
) -> dict[str, object]:
    if not model.corners:
        return {
            "surveyed_at": surveyed_at,
            "status": "PARTIAL",
            "survey_complete": False,
            "survey_type": "event_injected_full_perimeter_poc",
            "failure_reason": "canonical_corners_missing",
            "canonical_fence_model": {
                "status": "PARTIAL",
                "corners": {},
                "fences": {},
                "reference_points": {},
                "validation_errors": model.validation_errors,
            },
            "navigation_points": [
                {"label": p.label, "x_m": p.point.x_m, "y_m": p.point.y_m}
                for p in route
            ],
            "navigation_pattern": {
                "complete": False,
                "valid": False,
                "errors": model.validation_errors,
            },
        }

    xs = [p.x_m for p in model.corners.values()]
    ys = [p.y_m for p in model.corners.values()]
    west_x, east_x = min(xs), max(xs)
    south_y, north_y = min(ys), max(ys)

    court_half_width = court_lines.doubles_width_m / 2
    court_half_length = court_lines.length_m / 2
    measured_outer_width = court_lines.doubles_width_m
    is_doubles = abs(measured_outer_width - court_lines.doubles_width_m) <= court_lines.width_tolerance_m

    boundary_distances = {
        "near_baseline_to_fence_m": round(abs(south_y - (-court_half_length)), 3),
        "far_baseline_to_fence_m": round(abs(north_y - court_half_length), 3),
        "left_sideline_to_fence_m": round(abs(west_x - (-court_half_width)), 3),
        "right_sideline_to_fence_m": round(abs(east_x - court_half_width), 3),
    }

    net_ref = model.reference_points.get("near_net_standoff")
    net_world_pos = {
        "x_m": 0.0 if net_ref is None else round(net_ref.x_m, 3),
        "y_m": 0.0,
    }
    canonical_model = {
        "status": model.status,
        "corners": {
            label: {"x_m": round(point.x_m, 3), "y_m": round(point.y_m, 3)}
            for label, point in model.corners.items()
        },
        "fences": {
            label: {
                "start_corner": fence.start_corner,
                "end_corner": fence.end_corner,
                "length_m": round(fence.length_m, 3),
            }
            for label, fence in model.fences.items()
        },
        "reference_points": {
            label: {"x_m": round(point.x_m, 3), "y_m": round(point.y_m, 3)}
            for label, point in model.reference_points.items()
        },
        "validation_errors": model.validation_errors,
    }

    return {
        "surveyed_at": surveyed_at,
        "status": "SUCCESS" if model.status == "VALID" else "PARTIAL",
        "survey_complete": model.status == "VALID",
        "survey_type": "event_injected_full_perimeter_poc",
        "failure_reason": None if model.status == "VALID" else ";".join(model.validation_errors),
        "point_count": len(route),
        "is_doubles": is_doubles,
        "canonical_fence_model": canonical_model,
        "boundary_distances": boundary_distances,
        "court_geometry": {
            "length_m": court_lines.length_m,
            "width_m": measured_outer_width,
            "singles_width_m": court_lines.singles_width_m,
            "doubles_width_m": court_lines.doubles_width_m,
            "method": "canonical_event_injected_survey",
        },
        "geometry": {
            "net_world_pos": net_world_pos,
            "court_outer_lines": {
                "west_x": -court_half_width,
                "east_x": court_half_width,
                "south_y": -court_half_length,
                "north_y": court_half_length,
            },
        },
        "navigation_points": [
            {
                "label": p.label,
                "x_m": round(p.point.x_m, 3),
                "y_m": round(p.point.y_m, 3),
                "source_event": p.source_event.value,
                "confidence": p.confidence,
                "note": p.note,
            }
            for p in route
        ],
        "navigation_route": [
            {"label": p.label, "x_m": round(p.point.x_m, 3), "y_m": round(p.point.y_m, 3)}
            for p in route
        ],
        "navigation_pattern": {
            "complete": model.status == "VALID",
            "valid": model.status == "VALID",
            "matched": [p.label for p in route],
            "errors": model.validation_errors,
        },
    }


runtime_result = build_runtime_survey_result(model, survey.route)
print("runtime status:", runtime_result["status"])
print("survey_complete:", runtime_result["survey_complete"])
print("is_doubles:", runtime_result["is_doubles"])
print("boundary_distances:", runtime_result["boundary_distances"])
print("canonical corners:", runtime_result["canonical_fence_model"]["corners"])


## Regression for the first failure

This deliberately injects an early/short near-baseline corner. The section accepts the event because the semantic label is correct, but canonical validation rejects it because the baseline-side traverse was too short. This catches the failure where the robot did not drive the whole side to the upper-left corner.


In [ ]:
bad_survey = run_survey(early_corner_failure_events())
bad_model = CanonicalConverter().convert(bad_survey.route)
print("\nstatus:", bad_model.status)
print("validation_errors:", bad_model.validation_errors)
assert bad_model.status == "PARTIAL"
assert "near_baseline_to_left_corner_traverse_too_short" in bad_model.validation_errors

bad_fig = visualize_canonical_model(bad_survey.route, bad_model, "Synthetic early-corner failure")


## Where collision avoidance belongs

Collision avoidance should be added as a command supervisor after the event-driven route works. Each section proposes a `BaseCommand`; a safety layer then clamps, stops, or replaces that command based on front/side obstacle sectors, human detection, net-post proximity, stuck detection, and narrow-gap constraints.

That keeps route logic and safety logic separate: route sections answer "what am I trying to do?", while the safety supervisor answers "is this command safe right now?".
